# numbaによる実行時コンパイル

## インストール
Anaconda 2023.09では
numba
(https://numba.pydata.org/)
のupdateが必要かもしれません。
```
pip install -U numba
```

## 用途

Pythonでプログラムを作る際に
Pythonのloopは低速なのがproduction runで問題となるでしょう。
Pythonのloopを高速化する手法にnumbaによる実行時コンパイルを行う手法があります。



## 例

行列積を例とします。

In [1]:
import numpy as np
import time
from numba import jit

# (N,N)行列

N=400
A = np.random.rand(N,N)
B = np.random.rand(N,N)

In [2]:
result = [] # 結果を貯める

In [3]:
def AxB(A,B,N):
    C1 = np.zeros(N*N).reshape(N,N)
    for i in range(N):
        for k in range(N):
            c = 0
            for j in range(N):
                c += A[i,j]*B[j,k]
            C1[i,k] = c
    return C1

t1 = time.time()
C1 = AxB(A,B,N)
t2 = time.time()
result.append(["AxB", t2-t1, 1])
print(t2-t1,"sec")

15.95905613899231 sec


In [4]:
@jit(nopython=True)
def AxBjit(A,B,N):
    C = np.zeros(N*N).reshape(N,N)
    for i in range(N):
        for k in range(N):
            c = 0
            for j in range(N):
                c += A[i,j]*B[j,k]
            C[i,k] = c
    return C


In [5]:
for i in range(5):
    t1 = time.time()
    C2 = AxBjit(A,B,N)
    t2 = time.time()
    print(t2-t1,"sec.", "same result?", np.all(C2==C1))
    result.append(["AxB_jit", t2-t1, i+1])

0.5525553226470947 sec. same result? True
0.036707401275634766 sec. same result? True
0.03730440139770508 sec. same result? True
0.05231070518493652 sec. same result? True
0.04263615608215332 sec. same result? True


In [6]:
# blas使用
t1 = time.time()
C3 = np.dot(A,B)
t2 = time.time()
result.append(["blas3", t2-t1, 1]) 
print(t2-t1,"sec")

0.00844573974609375 sec


In [7]:
# 結果の表示
import pandas as pd
df = pd.DataFrame(result, columns=["method","elapsed_time","counter"])
df["speedup"] = df.loc[0,"elapsed_time"]/df["elapsed_time"].values
df

,method,elapsed_time,counter,speedup
0,AxB,15.959056,1,1.000000
1,AxB_jit,0.552555,1,28.882277
2,AxB_jit,0.036707,2,434.763987
3,AxB_jit,0.037304,3,427.806252
4,AxB_jit,0.052311,4,305.082030
5,AxB_jit,0.042636,5,374.308043
6,blas3,0.008446,1,1889.598380


一回目はコンパイルを行うので時間がかかりますが、
私のPCではnumbaを用いることで250倍程度の速度向上しました。
(blas level3を用いると更にその8000倍高速に動作します。)

## if文が入った行列積

内部にif文があるとblas level3は使えないので
簡単なif文を入れて速度比を測ります。

In [8]:
result_if = []

In [9]:
def AxB_if(A,B,N):
    C1 = np.zeros(N*N).reshape(N,N)
    for i in range(N):
        for k in range(N):
            c = 0
            for j in range(N):
                if A[i,j]>B[j,k]:
                    c += A[i,j]*B[j,k]
            C1[i,k] = c
    return C1

t1 = time.time()
C1_if = AxB(A,B,N)
t2 = time.time()
result_if.append(["AxB_if", t2-t1, 1])
print(t2-t1,"sec")

13.45113468170166 sec


In [10]:
@jit(nopython=True)
def AxBif_jit(A,B,N):
    C = np.zeros(N*N).reshape(N,N)
    for i in range(N):
        for k in range(N):
            c = 0
            for j in range(N):
                if A[i,j]>B[j,k]:
                    c += A[i,j]*B[j,k]
            C[i,k] = c
    return C


In [11]:
for i in range(5):
    t1 = time.time()
    C2_if = AxBjit(A,B,N)
    t2 = time.time()
    print(t2-t1,"sec.", "same result?", np.all(C2_if==C1_if))
    result_if.append(["AxB_if_jit", t2-t1, i+1])

0.04567313194274902 sec. same result? True
0.04084205627441406 sec. same result? True
0.039962053298950195 sec. same result? True
0.038400888442993164 sec. same result? True


0.03700971603393555 sec. same result? True


In [12]:
# 結果の表示
df_if = pd.DataFrame(result_if, columns=["method","elapsed_time","counter"])
df_if["speedup"] = df_if.loc[0,"elapsed_time"]/df_if["elapsed_time"].values
df_if

,method,elapsed_time,counter,speedup
0,AxB_if,13.451135,1,1.000000
1,AxB_if_jit,0.045673,1,294.508699
2,AxB_if_jit,0.040842,2,329.345188
3,AxB_if_jit,0.039962,3,336.597686
4,AxB_if_jit,0.038401,4,350.281861
5,AxB_if_jit,0.037010,5,363.448741


私のPCでは250倍以上程度の速度向上しました。

# 問題
上のコードで@jit(nopython=True)
を@jitとして実行せよ。NumbaDeprecationWarningが出力されるはずである。

LLMにこの警告メッセージの修正法を尋ねよ。
